In [9]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, fbeta_score, average_precision_score

In [10]:
# Define features and target variable
features = [
    # encoded station/service
    'StationCode_TE',
    'Service:Type_Intercity',
    'Service:Type_Intercity direct',
    'Service:Type_Sprinter',
    # temporal
    'Hour_sin',
    'Hour_cos',
    'DayOfWeek_sin',
    'DayOfWeek_cos',
    'Month_sin',
    'Month_cos',
    'IsWeekend',
    'RushHour',
    # operational context
    'StationTraffic',
    'Stop:Platform change',
    'arrival_scheduled',
    'departure_scheduled',
    # weather
    'Wind Direction',
    'Hourly Mean Wind Speed',
    'Wind Speed last 10 Minutes',
    'Max Wind Speed',
    'Temperature',
    'Dew Point temperature',
    'Sunshine Duration',
    'Global Radiation',
    'Precipitation Duration',
    'Precipitation Amount',
    'Air Pressure',
    'Horizontal Visibility',
    'Cloud Cover',
    'Humidity',
    'Fog',
    'Rainfall',
    'Snowfall',
    'Thunder',
    'Hail',
]

target = "is_cancelled"

In [11]:
# Prepare for chunked reading
cols = features + [target]
chunk_size = 1_000_000
sample_size = 1_000_000
random_state = 42

dtype_map = {col: "float32" for col in features}
dtype_map[target] = "int8"


# Read CSV in chunks
def chunk_reader(file_path):
    for chunk in pd.read_csv(
        file_path,
        usecols=cols,
        dtype=dtype_map,
        chunksize=chunk_size
    ):
        chunk = chunk.reindex(columns=cols, fill_value=0) # Ensure all columns are present

        X_chunk = chunk[features]
        y_chunk = chunk[target]

        yield X_chunk, y_chunk # Yield the chunk for processing
        
# Count classes
not_cancelled_total = 0
cancelled_total = 0

for X_chunk, y_chunk in chunk_reader("train_data.csv"):
    not_cancelled_total += (y_chunk == 0).sum()
    cancelled_total += (y_chunk == 1).sum()

total_rows = not_cancelled_total + cancelled_total

print(f"Train not cancelled: {not_cancelled_total:,}")
print(f"Train cancelled: {cancelled_total:,}")
print(f"Train total:     {total_rows:,}")

# Decide sample sizes
not_cancelled_sample_size = int(sample_size * not_cancelled_total / total_rows)
cancelled_sample_size = sample_size - not_cancelled_sample_size

print(f"Sampling not cancelled: {not_cancelled_sample_size:,}")
print(f"Sampling cancelled: {cancelled_sample_size:,}")

Train not cancelled: 43,903,090
Train cancelled: 4,479,366
Train total:     48,382,456
Sampling not cancelled: 907,417
Sampling cancelled: 92,583


In [12]:
# Generate random numbers for sampling (default_rng is faster)
rng = np.random.default_rng(random_state)

X_not_cancelled_parts = []
y_not_cancelled_parts = []
X_cancelled_parts = []
y_cancelled_parts = []

not_cancelled_seen = 0
cancelled_seen = 0

for X_chunk, y_chunk in chunk_reader("train_data.csv"):

    # Create masks for cancelled and not cancelled
    not_cancelled_mask = y_chunk == 0
    cancelled_mask = y_chunk == 1
    
    # Split the chunk into cancelled and not cancelled
    X_not_cancelled = X_chunk.loc[not_cancelled_mask]
    y_not_cancelled = y_chunk.loc[not_cancelled_mask]

    X_cancelled = X_chunk.loc[cancelled_mask]
    y_cancelled = y_chunk.loc[cancelled_mask]

    # sample fraction based on remaining needed / remaining available
    neg_remaining_needed = not_cancelled_sample_size - sum(len(p) for p in y_not_cancelled_parts)
    pos_remaining_needed = cancelled_sample_size - sum(len(p) for p in y_cancelled_parts)

    neg_remaining_total = not_cancelled_total - not_cancelled_seen
    pos_remaining_total = cancelled_total - cancelled_seen

    # Sample not cancelled
    if neg_remaining_needed > 0 and len(X_not_cancelled) > 0:
        p_neg = min(1.0, neg_remaining_needed / max(neg_remaining_total, 1))
        keep_neg = rng.random(len(X_not_cancelled)) < p_neg

        X_not_cancelled_parts.append(X_not_cancelled.loc[keep_neg])
        y_not_cancelled_parts.append(y_not_cancelled.loc[keep_neg])
        
    # Sample cancelled
    if pos_remaining_needed > 0 and len(X_cancelled) > 0:
        p_pos = min(1.0, pos_remaining_needed / max(pos_remaining_total, 1))
        keep_pos = rng.random(len(X_cancelled)) < p_pos

        X_cancelled_parts.append(X_cancelled.loc[keep_pos])
        y_cancelled_parts.append(y_cancelled.loc[keep_pos])

    not_cancelled_seen += len(X_not_cancelled)
    cancelled_seen += len(X_cancelled)

# Combine sampled parts
X_train_sample = pd.concat(X_not_cancelled_parts + X_cancelled_parts, axis=0)
y_train_sample = pd.concat(y_not_cancelled_parts + y_cancelled_parts, axis=0)

# Shuffle final sample
shuffle_idx = rng.permutation(len(X_train_sample))
X_train_sample = X_train_sample.iloc[shuffle_idx].to_numpy(dtype=np.float32, copy=False)
y_train_sample = y_train_sample.iloc[shuffle_idx].to_numpy(copy=False)

In [ ]:
param_dist = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [10, 12, 15, 18, 20, None],
    'min_samples_split': [5, 10, 20, 50, 100],
    'min_samples_leaf': [2, 5, 10, 20, 50],
    'max_features': ['sqrt', 'log2', 0.3, 0.5, 0.7],
    'class_weight': [
        'balanced',
        'balanced_subsample',
        {0: 1, 1: 5},
        {0: 1, 1: 10},
        {0: 1, 1: 15},
        {0: 1, 1: 20}
    ],
    'criterion': ['gini', 'entropy']
}

rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=15, 
    min_samples_split=20,
    min_samples_leaf=10,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train_sample, y_train_sample)
print(f"Random Forest trained on {len(y_train_sample):,} samples (stratified)")

# For sampling 
# from imblearn.ensemble import BalancedBaggingClassifier
# balanced_rf = BalancedBaggingClassifier(
#     RandomForestClassifier(n_estimators=50, max_depth=15),
#     sampling_strategy='auto',
#     replacement=True,
#     n_estimators=10
# )

Random Forest trained on 999,797 samples (stratified)


In [17]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import make_scorer, fbeta_score
import numpy as np

# Custom F2 scorer
f2_scorer = make_scorer(fbeta_score, beta=2)

# Expanded parameter grid for imbalance
param_dist = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [10, 12, 15, 18, 20, None],
    'min_samples_split': [5, 10, 20, 50, 100],
    'min_samples_leaf': [2, 5, 10, 20, 50],
    'max_features': ['sqrt', 'log2', 0.3, 0.5, 0.7],
    'class_weight': [
        'balanced',
        'balanced_subsample',
        {0: 1, 1: 5},
        {0: 1, 1: 10},
        {0: 1, 1: 15},
        {0: 1, 1: 20}
    ],
    'criterion': ['gini', 'entropy']
}

# Use smaller subset for tuning (20-30% of your data)
from sklearn.model_selection import train_test_split
X_tune, X_val_tune, y_tune, y_val_tune = train_test_split(
    X_train_sample, y_train_sample, 
    train_size=0.3,  # Use 30% for faster tuning
    stratify=y_train_sample,
    random_state=42
)

# Random search
rf_tuned = RandomizedSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_distributions=param_dist,
    n_iter=50,  # Try 50 combinations
    scoring=f2_scorer,
    cv=3,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

rf_tuned.fit(X_tune, y_tune)

print(f"Best F2 score: {rf_tuned.best_score_:.4f}")
print(f"Best parameters: {rf_tuned.best_params_}")

Fitting 3 folds for each of 50 candidates, totalling 150 fits
Best F2 score: 0.4150
Best parameters: {'n_estimators': 200, 'min_samples_split': 50, 'min_samples_leaf': 20, 'max_features': 0.3, 'max_depth': None, 'criterion': 'entropy', 'class_weight': {0: 1, 1: 20}}


In [18]:
# Evaluate CSV split in chunks
def evaluate_split_rf(name, file_path):
    y_true_parts = []
    y_pred_parts = []
    y_prob_parts = []

    for X_chunk, y_chunk in chunk_reader(file_path):
        X_chunk_np = X_chunk.to_numpy(dtype=np.float32, copy=False)

        y_true_parts.append(y_chunk.to_numpy(copy=False))
        y_pred_parts.append(rf.predict(X_chunk_np))
        y_prob_parts.append(rf.predict_proba(X_chunk_np)[:, 1])

    y_true_all = np.concatenate(y_true_parts)
    y_pred_all = np.concatenate(y_pred_parts)
    y_prob_all = np.concatenate(y_prob_parts)

    f2 = fbeta_score(y_true_all, y_pred_all, beta=2)

    print(f"\n{name}")
    print("F2 Score:", round(f2, 4))
    print(classification_report(y_true_all, y_pred_all, digits=3, zero_division=0))
    print("PR-AUC:", round(average_precision_score(y_true_all, y_prob_all), 4))


evaluate_split_rf("Validation", "val_data.csv")
evaluate_split_rf("Test", "test_data.csv")


Validation
F2 Score: 0.3212
              precision    recall  f1-score   support

           0      0.923     0.784     0.848   9333880
           1      0.173     0.409     0.243   1033789

    accuracy                          0.746  10367669
   macro avg      0.548     0.596     0.545  10367669
weighted avg      0.848     0.746     0.787  10367669

PR-AUC: 0.1649

Test
F2 Score: 0.3664
              precision    recall  f1-score   support

           0      0.909     0.764     0.831   9105889
           1      0.209     0.451     0.286   1261781

    accuracy                          0.726  10367670
   macro avg      0.559     0.608     0.558  10367670
weighted avg      0.824     0.726     0.764  10367670

PR-AUC: 0.2115
